# 직접 수집 2 · 정제 — 네트워크 필터링

확장된 네트워크에는 분석에 방해되는 **노이즈 문서**가 섞여 있습니다.
- `List of …`, `Lists of …` 같은 **목록성 문서**
- `data/wiki_rule.xlsx` 에 정의된 **제외 규칙**(title/category 키워드)

무거운 XTools 수집(다음 노트북) **전에** 걸러서, 불필요한 크롤링을 줄입니다.

In [ ]:
# --- 공통 준비: 프로젝트 루트로 이동 + 임포트 + 경로 정의 ---
import os, sys, warnings
warnings.filterwarnings("ignore")
for _ in range(5):                       # pipeline.py 있는 프로젝트 루트 탐색
    if os.path.isfile("pipeline.py"):
        break
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import pandas as pd
import pipeline as P
import wiki_crawling as wc

SEED = "Quantum computing"               # ← 분석할 위키 문서 (자유 변경)
N_DEPTH = 1                              # 1차수 고정
slug = P.slugify_seed(SEED)
BASE = os.path.join("runs", slug)
os.makedirs(os.path.join(BASE, "seed_item"), exist_ok=True)
os.makedirs(os.path.join(BASE, "xtools_item", slug), exist_ok=True)
EXPAND   = os.path.join(BASE, "seed_item", f"{N_DEPTH}차시 확장 최종_결과.xlsx")
FILTER   = os.path.join(BASE, "seed_item", f"{slug}_filtering_network.xlsx")
XTOOLS   = os.path.join(BASE, "xtools_item", slug)
FLAG     = os.path.join(XTOOLS, ".collect_done")
PAGERANK = os.path.join(BASE, f"{slug}_pagerank.xlsx")
STATS    = os.path.join(BASE, f"{slug}_statistics.xlsx")
print("작업 루트:", os.getcwd())
print("대상 문서:", SEED, "| 작업 폴더:", BASE)

In [ ]:
# [이전 단계 재현] 시드 실제 제목 (캐시되어 즉시)
df_true = P.resolve_true_title(SEED, os.path.join(BASE, "true_title.xlsx"))
SEED_TRUE = P.title_space(str(df_true.loc[0, "True_title"]))
print("시드 실제 제목:", SEED_TRUE)

In [ ]:
# [이전 단계 재현] 네트워크 확장 (캐시되어 즉시)
EXPAND = P.expand_network(SEED_TRUE, EXPAND, N_DEPTH)
print("확장 결과:", EXPAND)

## 필터링 실행 — 전/후 비교

In [ ]:
before = pd.read_excel(EXPAND)
before_nodes = set(before["From_title"].dropna()) | set(before["To_seealso"].dropna())

FILTER = P.filter_network(EXPAND, SEED_TRUE, FILTER, mode="balanced")

after = pd.read_excel(FILTER)
after_nodes = set(after["From_title"].dropna()) | set(after["To_seealso"].dropna())

print(f"엣지  : {len(before):3d}  →  {len(after):3d}")
print(f"노드  : {len(before_nodes):3d}  →  {len(after_nodes):3d}")
removed = before_nodes - after_nodes
print(f"제외된 문서 {len(removed)}개:")
for x in sorted(removed):
    print("  -", x)

## 남은 연관 문서 (수집 대상)

In [ ]:
after.head(20)

**정리**
- 목록성/규칙 문서를 제외해 **분석 가치가 있는 노드만** 남겼습니다.
- 다음 노트북(**03 수집②**)에서 이 노드들의 편집·조회·링크·정보를 수집합니다.